# Análise de Dados de Conversas do WhatsApp

Lê uma conversa de grupo exportada pelo WhatsApp (**sem mídia**), em `.txt` ou no `.zip` gerado pelo app, e oferece:

1. Resumo das conversas
2. Histórico do remetente
3. Gráfico do histórico do remetente (histograma por dia)
4. Gráfico de pizza
5. Gráfico de linhas

Formatos suportados: Android (`dd/mm/aaaa hh:mm - Nome: msg`) e iPhone (`[dd/mm/aaaa, hh:mm:ss] Nome: msg`).

In [ ]:
import logging
import re
import warnings
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 120)

# Fontes alternativas para nomes com emoji nos gráficos; silencia avisos de fonte ausente
plt.rcParams["font.family"] = ["DejaVu Sans", "Segoe UI Emoji", "Segoe UI Symbol", "Noto Emoji", "Symbola"]
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="Glyph .* missing from")

## Configuração
Coloque o arquivo exportado (`.zip` ou `.txt`) na mesma pasta do notebook. Se `ARQUIVO` ficar como `None`, o notebook usa automaticamente a conversa do WhatsApp encontrada na pasta; para escolher outro arquivo, informe o caminho.

In [ ]:
ARQUIVO = None  # ex.: "Conversa do WhatsApp com Meu Grupo.zip"

## Leitura do arquivo

In [ ]:
# Android: 23/03/2026 15:28 - Nome: mensagem
RE_ANDROID = re.compile(r"^(\d{1,2}/\d{1,2}/\d{2,4}),? (\d{1,2}:\d{2}(?::\d{2})?)\s*(?:[AaPp]\.?\s?[Mm]\.?)?\s-\s(.*)$")
# iPhone: [23/03/2026, 15:28:10] Nome: mensagem
RE_IPHONE = re.compile(r"^\[(\d{1,2}/\d{1,2}/\d{2,4}),? (\d{1,2}:\d{2}(?::\d{2})?)\s*(?:[AaPp]\.?\s?[Mm]\.?)?\]\s(.*)$")
CONTROLE = re.compile(r"[‎‏‪-‮⁦-⁩﻿]")


def localizar_arquivo(arquivo=None):
    if arquivo:
        caminho = Path(arquivo)
        if not caminho.exists():
            raise FileNotFoundError(f"Arquivo não encontrado: {caminho.resolve()}")
        return caminho
    candidatos = sorted(p for p in Path.cwd().iterdir()
                        if p.suffix.lower() in (".zip", ".txt") and "whatsapp" in p.name.lower())
    if not candidatos:
        raise FileNotFoundError(f"Nenhuma conversa do WhatsApp (.zip/.txt) encontrada em {Path.cwd()}. "
                                "Copie o arquivo exportado para esta pasta ou preencha ARQUIVO.")
    return candidatos[0]


def ler_texto(caminho):
    if caminho.suffix.lower() == ".zip":
        with zipfile.ZipFile(caminho) as z:
            txts = [n for n in z.namelist() if n.lower().endswith(".txt")]
            if not txts:
                raise ValueError(f"O arquivo {caminho.name} não contém a conversa (.txt).")
            return z.read(txts[0]).decode("utf-8-sig", errors="replace")
    return caminho.read_text(encoding="utf-8-sig", errors="replace")


def carregar_conversa(caminho):
    registros = []
    for linha in ler_texto(caminho).splitlines():
        linha = CONTROLE.sub("", linha)
        m = RE_ANDROID.match(linha) or RE_IPHONE.match(linha)
        if m:
            data, hora, resto = m.groups()
            if ": " in resto:
                remetente, mensagem = resto.split(": ", 1)
                registros.append([data, hora, remetente.strip().lstrip("~").strip(), mensagem])
            else:
                registros.append(None)  # mensagem do sistema (criou grupo, adicionou etc.)
        elif registros and registros[-1] is not None:
            registros[-1][3] += "\n" + linha  # continuação de mensagem com várias linhas

    df = pd.DataFrame([r for r in registros if r], columns=["data", "hora", "remetente", "mensagem"])
    if df.empty:
        raise ValueError("Nenhuma mensagem reconhecida. Verifique se o arquivo é uma conversa exportada do WhatsApp.")
    df["mensagem"] = df["mensagem"].str.rstrip()
    df["data"] = pd.to_datetime(df["data"], dayfirst=True, format="mixed").dt.date
    df["hora"] = df["hora"].str.slice(0, 5)
    return df


caminho = localizar_arquivo(ARQUIVO)
df = carregar_conversa(caminho)
print(f"Arquivo: {caminho.name}")
print(f"{len(df)} mensagens | {df['remetente'].nunique()} remetentes | {df['data'].min()} a {df['data'].max()}")

## Tabela completa

In [ ]:
df

## 1. Resumo das conversas
Total de mensagens por remetente, do maior para o menor.

In [ ]:
def resumo_conversas(df):
    return (df["remetente"].value_counts()
              .rename_axis("remetente").reset_index(name="total_mensagens"))

resumo_conversas(df)

## 2. Histórico do remetente
Selecione o remetente para ver suas mensagens com data e hora.

In [ ]:
def historico_remetente(df, remetente):
    return df.loc[df["remetente"] == remetente, ["data", "hora", "mensagem"]].reset_index(drop=True)

remetentes = resumo_conversas(df)["remetente"].tolist()
widgets.interact(lambda remetente: display(historico_remetente(df, remetente)),
                 remetente=widgets.Dropdown(options=remetentes, description="Remetente:"));

## 3. Gráfico do histórico do remetente
Histograma da quantidade de mensagens por dia. Primeiro o seletor individual, depois todos os remetentes.

In [ ]:
def histograma_remetente(df, remetente, ax=None):
    datas = pd.to_datetime(df.loc[df["remetente"] == remetente, "data"])
    todas = pd.to_datetime(df["data"])
    dias = pd.date_range(todas.min(), todas.max() + pd.Timedelta(days=1))  # um bin por dia, período da conversa inteira
    ax = ax or plt.subplots(figsize=(12, 4))[1]
    ax.hist(datas, bins=dias, color="#2a7ab0", edgecolor="white")
    ax.set_xlim(dias[0], dias[-1])
    ax.set_title(f"{remetente} — {len(datas)} mensagens")
    ax.set_xlabel("Data")
    ax.set_ylabel("Mensagens por dia")
    ax.tick_params(axis="x", rotation=45)
    return ax

def mostrar_histograma(remetente):
    histograma_remetente(df, remetente)
    plt.tight_layout()
    plt.show()

widgets.interact(mostrar_histograma,
                 remetente=widgets.Dropdown(options=remetentes, description="Remetente:"));

In [ ]:
n = len(remetentes)
fig, axes = plt.subplots(n, 1, figsize=(12, 3.2 * n), squeeze=False)
for ax, r in zip(axes[:, 0], remetentes):
    histograma_remetente(df, r, ax)
plt.tight_layout()
plt.show()

## 4. Gráfico de pizza
Percentual de mensagens de cada remetente.

In [ ]:
def grafico_pizza(df):
    contagem = df["remetente"].value_counts()
    pct = contagem / contagem.sum() * 100
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.pie(contagem, autopct=lambda p: f"{p:.1f}%" if p >= 4 else "", startangle=90,
           counterclock=False, pctdistance=0.75, wedgeprops={"edgecolor": "white"})
    ax.legend([f"{r} ({v:.1f}%)" for r, v in pct.items()], bbox_to_anchor=(1, 0.5), loc="center left")
    ax.set_title("Percentual de mensagens por remetente")
    plt.show()

grafico_pizza(df)

## 5. Gráfico de linhas
Quantidade de mensagens por data, uma linha (cor) por remetente.

In [ ]:
def grafico_linhas(df):
    tabela = (df.groupby([pd.to_datetime(df["data"]), "remetente"]).size()
                .unstack(fill_value=0))
    tabela = tabela.reindex(pd.date_range(tabela.index.min(), tabela.index.max()), fill_value=0)
    tabela = tabela[resumo_conversas(df)["remetente"]]  # legenda na ordem de volume

    cores = plt.cm.tab10.colors + plt.cm.Dark2.colors
    fig, ax = plt.subplots(figsize=(14, 6))
    for i, r in enumerate(tabela.columns):
        ax.plot(tabela.index, tabela[r], label=r, color=cores[i % len(cores)], linewidth=1.5)
    ax.set_title("Mensagens ao longo do tempo por remetente")
    ax.set_xlabel("Data")
    ax.set_ylabel("Quantidade de mensagens")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

grafico_linhas(df)

## Menu interativo (opcional)
Executa qualquer uma das análises acima a partir de um único seletor.

In [ ]:
saida = widgets.Output()
opcao = widgets.Dropdown(description="Análise:", style={"description_width": "initial"}, options=[
    ("1 - Resumo das conversas", 1), ("2 - Histórico do remetente", 2),
    ("3 - Gráfico do histórico do remetente", 3), ("4 - Gráfico de pizza", 4), ("5 - Gráfico de linhas", 5)])
rem = widgets.Dropdown(options=remetentes, description="Remetente:")
botao = widgets.Button(description="Executar", button_style="primary")

def executar(_):
    with saida:
        saida.clear_output()
        if opcao.value == 1:
            display(resumo_conversas(df))
        elif opcao.value == 2:
            display(historico_remetente(df, rem.value))
        elif opcao.value == 3:
            mostrar_histograma(rem.value)
        elif opcao.value == 4:
            grafico_pizza(df)
        else:
            grafico_linhas(df)

botao.on_click(executar)
display(widgets.VBox([widgets.HBox([opcao, rem, botao]), saida]))